# Part 2 — RFM Segmentation & Retention Strategy
## D2C Customer Churn Intelligence Capstone

**Objective:** Build customer segments using RFM + behavioural signals and recommend targeted retention actions.  
**Snapshot Date:** 2025-09-30 | **Universe:** 2,400 customers


## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#FAFAFA','axes.facecolor':'#FAFAFA',
    'axes.edgecolor':'#CCCCCC','axes.grid':True,'grid.color':'#E0E0E0',
    'grid.linestyle':'--','grid.alpha':0.7,'font.size':11,
    'axes.titlesize':13,'axes.titleweight':'bold',
})
PALETTE = ['#2E86AB','#E84855','#F4A261','#57CC99','#9B5DE5','#F72585','#FFB703','#023E8A']
SNAP = pd.Timestamp('2025-09-30')

DATA_DIR = '../data/'   # adjust if needed

customers    = pd.read_csv(DATA_DIR + 'customers.csv')
orders       = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
tickets      = pd.read_csv(DATA_DIR + 'support_tickets.csv', parse_dates=['ticket_date'])
web          = pd.read_csv(DATA_DIR + 'web_events_snapshot.csv')
churn        = pd.read_csv(DATA_DIR + 'churn_labels.csv')
intervention = pd.read_csv(DATA_DIR + 'intervention_history.csv')

# Clean pre-snapshot orders (no DUPs, no future rows)
orders_pre = orders[
    (orders['order_date'] <= SNAP) &
    (~orders['order_id'].str.contains('_DUP', na=False))
].copy()

print(f"Clean pre-snapshot orders: {len(orders_pre)}")
print(f"Customers: {len(customers)}")


Clean pre-snapshot orders: 8128
Customers: 2400


## 2. Build RFM Features

In [2]:
# ── Core RFM aggregation ─────────────────────────────────────────────────
rfm = orders_pre.groupby('customer_id').agg(
    last_order_date=('order_date','max'),
    frequency=('order_id','count'),
    monetary=('gross_amount','sum')
).reset_index()

rfm['recency'] = (SNAP - rfm['last_order_date']).dt.days

# Quintile scoring 1-5 (5 = best for each dimension)
rfm['R'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Total'] = rfm['R'] + rfm['F'] + rfm['M']

print("RFM Feature Summary:")
display(rfm[['recency','frequency','monetary','R','F','M','RFM_Total']].describe().round(2))


RFM Feature Summary:


,recency,frequency,monetary,R,F,M,RFM_Total
count,2400.00,2400.00,2400.00,2400.00,2400.00,2400.00,2400.00
mean,87.38,3.39,2547.12,3.01,3.00,3.00,9.01
std,80.14,2.38,2127.69,1.42,1.41,1.41,3.10
min,0.00,1.00,149.00,1.00,1.00,1.00,3.00
25%,25.00,1.00,954.52,2.00,2.00,2.00,7.00
50%,66.00,3.00,2010.71,3.00,3.00,3.00,9.00
75%,129.00,5.00,3562.12,4.00,4.00,4.00,11.00
max,562.00,16.00,27215.92,5.00,5.00,5.00,15.00


## 3. Add Non-RFM Behavioural Signals

In [3]:
# ── Signal 1: Support tickets (quality + volume) ─────────────────────────
ticket_feats = tickets[tickets['ticket_date'] <= SNAP].groupby('customer_id').agg(
    ticket_count=('ticket_id','count'),
    avg_sentiment=('sentiment_score','mean'),
    reopened_count=('reopened','sum'),
    neg_ticket_rate=('sentiment_score', lambda x: (x < 0).mean())
).reset_index()

# ── Signal 2: Return rate & discount usage ────────────────────────────────
return_feats = orders_pre.groupby('customer_id').agg(
    return_rate=('returned','mean'),
    avg_discount=('discount_pct','mean')
).reset_index()

# ── Signal 3: Web/app activity ────────────────────────────────────────────
web_feats = web[['customer_id','sessions_30d','abandoned_carts_30d',
                 'last_visit_days_ago','campaign_clicks_30d','email_opens_30d']].copy()

# ── Merge everything ──────────────────────────────────────────────────────
base = customers[['customer_id','loyalty_tier','preferred_category',
                  'acquisition_channel','age_group','city_tier']].copy()
base = (base
    .merge(rfm[['customer_id','recency','frequency','monetary','R','F','M','RFM_Total']], on='customer_id', how='left')
    .merge(ticket_feats, on='customer_id', how='left')
    .merge(return_feats, on='customer_id', how='left')
    .merge(web_feats, on='customer_id', how='left')
    .merge(churn[['customer_id','churn_next_60d']], on='customer_id', how='left')
    .merge(intervention[['customer_id','last_campaign_received','manual_priority_bucket']], on='customer_id', how='left')
)

# Fill NaN for customers with no orders/tickets
for col, fill in [('recency',999),('frequency',0),('monetary',0),
                  ('R',1),('F',1),('M',1),('RFM_Total',3),
                  ('ticket_count',0),('avg_sentiment',0),('reopened_count',0),
                  ('neg_ticket_rate',0),('return_rate',0),('avg_discount',0)]:
    base[col] = base[col].fillna(fill)

print(f"Base table shape: {base.shape}")
print(f"Columns: {list(base.columns)}")


Base table shape: (2400, 27)
Columns: ['customer_id', 'loyalty_tier', 'preferred_category', 'acquisition_channel', 'age_group', 'city_tier', 'recency', 'frequency', 'monetary', 'R', 'F', 'M', 'RFM_Total', 'ticket_count', 'avg_sentiment', 'reopened_count', 'neg_ticket_rate', 'return_rate', 'avg_discount', 'sessions_30d', 'abandoned_carts_30d', 'last_visit_days_ago', 'campaign_clicks_30d', 'email_opens_30d', 'churn_next_60d', 'last_campaign_received', 'manual_priority_bucket']


## 4. Segment Assignment Logic

In [4]:
def assign_segment(row):
    r, f, m       = int(row['R']), int(row['F']), int(row['M'])
    rfm_total     = row['RFM_Total']
    recency       = row['recency']
    ticket_count  = row['ticket_count']
    avg_sentiment = row['avg_sentiment']
    return_rate   = row['return_rate']
    avg_discount  = row['avg_discount']
    sessions      = row['sessions_30d']
    last_visit    = row['last_visit_days_ago']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    if r >= 3 and f >= 3 and rfm_total >= 10:
        return 'Loyal Customers'
    if m >= 4 and (avg_sentiment < -0.1 or ticket_count >= 3 or return_rate > 0.4):
        return 'High-Value but Unhappy'
    if avg_discount >= 0.45 and rfm_total <= 9:
        return 'Discount Sensitive'
    if r <= 2 and (f >= 2 or m >= 2):
        return 'At Risk'
    if recency > 150 or (sessions <= 1 and last_visit > 20 and f <= 1):
        return 'Dormant'
    if f <= 1 and recency <= 60:
        return 'New Customers'
    return 'Needs Nurturing'

base['segment_name'] = base.apply(assign_segment, axis=1)

print("Segment counts:")
print(base['segment_name'].value_counts())
print("\nChurn rate per segment:")
print((base.groupby('segment_name')['churn_next_60d'].mean()*100).sort_values(ascending=False).round(1))


Segment counts:
segment_name
At Risk                   623
Needs Nurturing           451
Loyal Customers           421
Champions                 344
New Customers             213
High-Value but Unhappy    198
Discount Sensitive         88
Dormant                    62
Name: count, dtype: int64

Churn rate per segment:
segment_name
Dormant                   90.3
At Risk                   81.5
High-Value but Unhappy    73.7
Discount Sensitive        58.0
Needs Nurturing           40.1
Loyal Customers           24.0
New Customers             22.5
Champions                 10.5
Name: churn_next_60d, dtype: float64


## 5. Visualisation (Chart 1 — Segment Size & Churn Rate)

In [5]:
seg_order = ['Champions','Loyal Customers','New Customers','Needs Nurturing',
             'Discount Sensitive','High-Value but Unhappy','At Risk','Dormant']
seg_colors = dict(zip(seg_order, PALETTE))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

counts = base['segment_name'].value_counts().reindex(seg_order)
bars = axes[0].barh(counts.index, counts.values,
                    color=[seg_colors[s] for s in counts.index], edgecolor='white')
for bar, v in zip(bars, counts.values):
    axes[0].text(v+5, bar.get_y()+bar.get_height()/2, str(v), va='center', fontweight='bold')
axes[0].set_xlabel('Customer Count')
axes[0].set_title('Segment Size')

churn_rates = base.groupby('segment_name')['churn_next_60d'].mean().reindex(seg_order)*100
bars2 = axes[1].barh(churn_rates.index, churn_rates.values,
                     color=[seg_colors[s] for s in churn_rates.index], edgecolor='white')
for bar, v in zip(bars2, churn_rates.values):
    axes[1].text(v+0.5, bar.get_y()+bar.get_height()/2, f'{v:.1f}%', va='center', fontweight='bold')
axes[1].set_xlabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Segment')
axes[1].set_xlim(0, 110)

plt.tight_layout()
plt.savefig('chart1_segment_overview.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Segment Profile Heatmap (Chart 3)

In [6]:
profile_cols = ['recency','frequency','monetary','ticket_count',
                'return_rate','avg_discount','sessions_30d','last_visit_days_ago']
profile = base.groupby('segment_name')[profile_cols].mean().reindex(seg_order)
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(profile_norm, annot=profile.round(1), fmt='.1f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Normalised Value'})
ax.set_title('Segment Profile Heatmap (mean values, normalised for colour)')
plt.tight_layout()
plt.savefig('chart3_segment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Budget Prioritisation Chart (Chart 4)

In [7]:
seg_df = pd.DataFrame({
    'segment':     seg_order,
    'size':        [344, 421, 213, 451, 88, 198, 623, 62],
    'churn_rate':  [0.105, 0.240, 0.225, 0.401, 0.580, 0.737, 0.815, 0.903],
    'avg_monetary':[4916, 3354, 793, 1116, 596, 4456, 2202, 505],
})

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(seg_df['churn_rate']*100, seg_df['avg_monetary'],
                     s=seg_df['size']*0.7,
                     c=PALETTE[:len(seg_df)], alpha=0.8,
                     edgecolors='white', linewidths=1.5)
for _, row in seg_df.iterrows():
    ax.annotate(row['segment'],
                (row['churn_rate']*100, row['avg_monetary']),
                fontsize=8.5, ha='center', va='bottom',
                xytext=(0, 10), textcoords='offset points')
ax.set_xlabel('Churn Rate (%)')
ax.set_ylabel('Avg Monetary Value (INR)')
ax.set_title('Segment Prioritisation Matrix\n(bubble size = segment size)')
ax.axvline(50, color='gray', linestyle='--', alpha=0.5, label='50% churn threshold')
ax.axhline(1000, color='gray', linestyle='--', alpha=0.5, label='₹1,000 monetary threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('chart4_budget_priority.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Save segments.csv

In [8]:
segments_out = base[[
    'customer_id','segment_name','recency','frequency','monetary',
    'R','F','M','RFM_Total','ticket_count','neg_ticket_rate',
    'return_rate','avg_discount','sessions_30d','last_visit_days_ago',
    'campaign_clicks_30d','loyalty_tier','churn_next_60d'
]].copy()

segments_out.to_csv('segments.csv', index=False)
print(f"Saved segments.csv: {segments_out.shape}")
display(segments_out.head(5))


Saved segments.csv: (2400, 18)


,customer_id,segment_name,recency,frequency,monetary,R,F,M,RFM_Total,ticket_count,neg_ticket_rate,return_rate,avg_discount,sessions_30d,last_visit_days_ago,campaign_clicks_30d,loyalty_tier,churn_next_60d
0,CUST00001,At Risk,107,6,2955.57,2,5,4,11,2.0,0.5,0.166667,0.363333,1,20,0,Silver,1
1,CUST00002,New Customers,40,1,581.00,4,1,1,6,1.0,0.0,0.000000,0.230000,8,0,0,Silver,0
2,CUST00003,Discount Sensitive,171,1,649.98,1,1,1,3,0.0,0.0,0.000000,0.470000,1,26,0,NaN,1
3,CUST00004,At Risk,131,1,1604.04,2,1,3,6,0.0,0.0,0.000000,0.160000,1,14,0,NaN,1
4,CUST00005,Loyal Customers,38,4,2550.91,4,3,3,10,1.0,1.0,0.000000,0.442500,18,9,1,Gold,0
